# 4 Smart Charging Using Reinforcement Learning

The previous sections studied *where* and *when* ride-hailing demand occurs. This section turns to the operational side of an electric fleet: given that a vehicle must be ready for its shift, *how should it charge*? We leave the Chicago trip data behind and study a single electric taxi that charges at home. The driver arrives at 2 p.m. and leaves at 4 p.m., so there is a two-hour window in which a charging agent sets the charging power every 15 minutes, giving eight sequential decisions. The energy the vehicle will need for the coming day is uncertain and is revealed only at departure, drawn from a normal distribution. Charging cost grows exponentially with power, and running out of energy is heavily penalised.

This makes the problem a sequential decision under uncertainty, which we formalise as a Markov decision process and solve with reinforcement learning. The agent has to balance two opposing forces: charge enough to cover an uncertain demand (safety), while spreading the load and exploiting cheaper time slots to keep cost down (economy).

The questions guiding this section are:

1. How is home charging formalised as a Markov decision process (states, actions, reward)?
2. Which charging policy minimises cost while avoiding energy shortfall under uncertain demand?
3. How close to the provable optimum does a learned DQN policy get, and does it beat naive charging strategies?
4. How does the policy react to the electricity price structure and to the level of demand uncertainty?

The section is organised as follows:

- *4.1 Problem Formalisation (MDP)*
- *4.2 Environment Implementation*
- *4.3 Design Choices and Parameters*
- *4.4 Reinforcement Learning Solution (DQN)*
- *4.5 Results: Policy and Evaluation against Baselines*
- *4.6 Sensitivity and Discussion*

The charging demand is modelled synthetically and independently of the trip data; where useful we anchor its parameters to realistic daily driving so the setup stays grounded rather than arbitrary.

## 4.1 Problem Formalisation (MDP)

Before any code, the charging problem is written out as a complete Markov decision process. This formalisation is the backbone of the section: the environment in 4.2 and the agent in 4.4 must both implement exactly this definition. The following has to be specified:

- **State** `s_t = (t, SoC_t)`: the current 15-minute slot `t` in {0, ..., 7} and the battery state of charge `SoC_t` in kWh (discretised). The slot belongs in the state because the electricity price varies over time, so the best action depends on *when* we are.
- **Action** `a_t`: the charging power for the slot, taken from a small discrete set (zero, low, medium, high in kW) so that value-based methods such as DQN apply directly.
- **Transition:** charging is deterministic, `SoC_{t+1} = min(B, SoC_t + a_t * 0.25)` over a 15-minute slot, capped at battery capacity `B`. The only stochastic element is the daily energy demand `D ~ N(mu, sigma)`, drawn once at departure (after slot 7).
- **Reward:** the per-slot cost `-alpha_t * exp(a_t)` following the assignment's exponential cost, plus a terminal penalty `-P` applied when `SoC_final < D`, i.e. the vehicle cannot cover its day.
- **Horizon and discount:** a finite horizon of eight steps with discount `gamma = 1`, since every decision within one short session matters equally and there is no long future to discount.

State the Markov property explicitly (the state carries everything needed to decide, so the history is irrelevant) and list every assumption (fixed window, charging only, no driving in between, demand revealed only at the end). Present the MDP as one compact table so a reader grasps the whole decision problem at a glance.

## 4.2 Environment Implementation

Implement the MDP from 4.1 as a small, self-contained simulation with a clean interface, so the *same* environment can be driven by the RL agent and by every baseline. This is what makes the later comparison fair. The cell below has to deliver:

- A class exposing `reset()` and `step(action)` in the style of a Gym environment. `step` applies the action, updates the state of charge, accumulates the slot cost, advances the clock, and on the final step draws the demand `D` and adds the shortfall penalty if needed. It returns `(next_state, reward, done, info)`.
- A seedable random generator for the demand draw, so every run is reproducible.
- An `info` dictionary logging per-step cost, state of charge, the drawn demand and whether a shortfall occurred, so 4.5 can analyse trajectories without re-running the training.
- A sanity check: run one fixed dummy policy (for example always *medium*), print the resulting trajectory and final cost, and confirm that the dynamics, the cap at `B` and the penalty all behave exactly as specified in 4.1.

Keep the environment deliberately simple and free of any agent logic. All intelligence belongs in 4.4.

In [ ]:
# 4.2  Environment (Gym-style reset/step) + sanity check


## 4.3 Design Choices and Parameters

Fix and *justify* every parameter of the environment. This is where the problem is made non-trivial, so the choices are argued, not merely stated. This subsection has to provide:

- A parameter table: demand mean `mu` and standard deviation `sigma`, battery capacity `B`, maximum power and the discrete action levels in kW, the initial state of charge at 2 p.m., the shortfall penalty `P`, and the state-of-charge discretisation step.
- The electricity price profile `alpha_t` over the eight slots. It has to be a genuine time-of-use curve with clearly cheaper and more expensive slots. A flat profile would make charging timing irrelevant and leave nothing to learn, so a non-trivial profile is required, and its shape is justified (for example an evening peak).
- A feasibility argument: with a 22 kW cap over two hours at most 44 kWh can be added, so `B` and the initial charge are chosen such that covering `mu` plus a safety buffer is actually reachable inside the window.
- A calibration of the penalty `P` so that it strictly dominates the largest plausible charging cost, which guarantees the agent never trades safety for a few cents of savings.

Anchor `mu` and `sigma` to plausible daily driving (average daily distance times energy consumption per km) so the synthetic demand is defensible. This is the one place where the trip data from Tasks 1 to 3 can be borrowed to set a realistic average daily distance.

In [ ]:
# 4.3  Parameters, alpha_t price profile, feasibility + penalty calibration


## 4.4 Reinforcement Learning Solution (DQN)

Solve the MDP with a Deep Q-Network, the method the course covers for discrete-action control. This subsection has to deliver:

- A Q-network mapping the state to one Q-value per action, trained with epsilon-greedy exploration, an experience replay buffer and a target network. Report the architecture and all hyperparameters.
- Training over many episodes, each a fresh charging session with a newly drawn demand, plus a convergence plot (episode reward and loss) that demonstrates the agent actually learns rather than merely runs.
- A provable reference: because the state space is small, also compute the exact optimal policy by dynamic programming (value iteration over the discretised MDP). This optimum is the yardstick the DQN is measured against in 4.5 and is what lifts the evaluation from descriptive to rigorous.

Show the DQN learning curve approaching the dynamic-programming optimum, so convergence is quantified and not just asserted. A tabular Q-learning agent may be added as a lightweight second learner, but the DP optimum is the reference that matters.

In [ ]:
# 4.4  DQN agent (replay, target net) + exact DP optimum (value iteration)


## 4.5 Results: Policy and Evaluation against Baselines

Demonstrate that the learned policy is both safe and economical, and benchmark it properly. This subsection has to provide:

- A visualisation of the learned policy: the chosen action as a function of the state `(t, SoC)`, and the charging schedule of a representative episode, so the behaviour is readable rather than a black box.
- An evaluation over many independent test episodes on two metrics that capture the trade-off: mean recharging cost and shortfall rate (how often the vehicle runs out of energy). Reporting only one of the two is not enough.
- A benchmark table comparing the DQN against the dynamic-programming optimum and against naive baselines: constant charging, greedy charge-as-fast-as-possible, and a cheapest-slots heuristic. The DQN should sit close to the optimum and clearly dominate the naive strategies.
- Distributions, not only averages: show the spread of cost and the rare shortfall events, since a fleet operator cares about the worst cases, not just the mean.

This benchmarking layer is what turns "we built an RL agent" into "our agent is provably good", and it mirrors the evaluation rigour applied to the predictive models in Section 3.

In [ ]:
# 4.5  Policy visualisation + benchmark table (DQN vs DP optimum vs baselines)


## 4.6 Sensitivity and Discussion

Show that the result is robust and translate it into advice, with a few targeted experiments rather than an exhaustive grid. This subsection has to cover:

- Vary the electricity price profile from flat to steeply peaked and show how charging concentrates into the cheap slots as the profile steepens.
- Vary the demand uncertainty `sigma` and show that the policy keeps a larger safety buffer as uncertainty grows, making the safety-versus-cost trade-off explicit.
- Briefly vary the penalty `P` and the maximum power to confirm the policy still reacts sensibly.
- Interpret in plain terms what the agent has learned, then connect it to the business question of Section 5: when home charging is sufficient, and what this implies for the choice between private and public charging infrastructure.

Close by stating the limitations honestly (a single vehicle, a synthetic demand, a stylised price profile), so the scope of the conclusion is clear and not overstated.

In [ ]:
# 4.6  Sensitivity sweeps (price profile, sigma, penalty, max power) + discussion
